In [0]:
def write_post_content_table(spark, run_date, db, table_names):
    sdf = spark.sql("""
                    INSERT OVERWRITE TABLE {db}.{post_content} PARTITION (partition_date = '{run_date}')
                    select p.post_id, 
                    p.content as text_content, 
                    if(size(mi.pic_url) = 0, NULL, mi.pic_url) as pic_url,
                    if(size(mi.video_url) = 0, NULL, mi.video_url) as video_url
                    from (
                      select `id` as post_id, content
                      from datalake_community.dw_cust_community_tb_post p
                      where to_date(from_unixtime(p.post_time/1000)) = '{run_date}'
                    ) p
                    left join (
                      select post_id,
                      collect_list(`url`) filter (where file_type in (0,1,2,3)) as pic_url,
                      collect_list(`url`) filter (where file_type = 10) as video_url
                      from {db}.{media_item_index}
                      group by 1
                    ) mi
                    on p.post_id = mi.post_id
                    """.format(run_date = run_date,
                               db = db,
                               post_content = table_names["post_content"],
                               media_item_index = table_names["media_item_index"]))

In [0]:
import json

# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

config_path = dbutils.widgets.get("config_path")
with open(config_path, "r") as file:
    config = json.load(file)
db, table_names = config['DATABASE'], config['TABLE_NAMES']

In [0]:
write_post_content_table(spark, run_date, db, table_names)